In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
import glob

SCAN_FILES = sorted(glob.glob("out/scan*_stripped.csv.gz"))
print(f"Loading {len(SCAN_FILES)} files...")

frames = []
for f in SCAN_FILES:
    frames.append(pd.read_csv(f, keep_default_na=False, dtype={"size_bytes": str}))
df = pd.concat(frames, ignore_index=True)

files = df[df["error"] == ""].copy()
errors = df[df["error"] != ""].copy()

files["size_bytes"] = files["size_bytes"].astype(int)
files["size_tb"] = files["size_bytes"] / 1e12
files["mtime"] = pd.to_datetime(files["mtime"])

# users with at least one permission denied entry
denied_users = set(errors["owner"].unique())

print(f"Total files:        {len(files):,}")
print(f"Permission denied:  {len(errors):,}")
print(f"Total size:         {files['size_tb'].sum():.2f} TB")

In [ ]:
# Pie charts: by volume and by file count, per user
def label(owner):
    return owner + ("*" if owner in denied_users else "")

by_volume = files.groupby("owner")["size_tb"].sum().sort_values(ascending=False)
by_count  = files.groupby("owner")["size_bytes"].count().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, data, title, fmt in [
    (axes[0], by_volume, "Disk usage by volume (TB)", lambda v: f"{v:.1f} TB"),
    (axes[1], by_count,  "Disk usage by file count",  lambda v: f"{v/1e6:.1f}M"),
]:
    labels = [label(o) for o in data.index]
    wedges, texts, autotexts = ax.pie(
        data.values,
        labels=labels,
        autopct=lambda p: fmt(p / 100 * data.sum()),
        pctdistance=0.75,
        startangle=90,
    )
    for t in autotexts:
        t.set_fontsize(8)
    ax.set_title(title)

fig.suptitle("* = has permission-denied entries", fontsize=10, y=0.02)
fig.tight_layout()
plt.show()

In [ ]:
# Top 20 largest directories (first 3 path components)
files["directory"] = files["path"].apply(lambda p: "/".join(Path(p).parts[:4]))

by_dir = (
    files.groupby("directory")["size_tb"]
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(10, 6))
by_dir.plot.barh(ax=ax)
ax.invert_yaxis()
ax.set_xlabel("Total size (TB)")
ax.set_title("Top 20 largest directories")
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:.1f}"))
fig.tight_layout()
plt.show()

In [ ]:
# Per-user statistics table
stats = files.groupby("owner").agg(
    total_files=("size_bytes", "count"),
    total_tb=("size_tb", "sum"),
    avg_file_size_mb=("size_bytes", lambda x: x.mean() / 1e6),
    median_file_size_mb=("size_bytes", lambda x: x.median() / 1e6),
    largest_file_gb=("size_bytes", lambda x: x.max() / 1e9),
).sort_values("total_tb", ascending=False)

stats.index = [label(o) for o in stats.index]
stats.columns = ["Total files", "Total (TB)", "Avg file size (MB)", "Median file size (MB)", "Largest file (GB)"]
stats["Total files"] = stats["Total files"].apply(lambda x: f"{x:,}")
stats["Total (TB)"] = stats["Total (TB)"].apply(lambda x: f"{x:.2f}")
stats["Avg file size (MB)"] = stats["Avg file size (MB)"].apply(lambda x: f"{x:.1f}")
stats["Median file size (MB)"] = stats["Median file size (MB)"].apply(lambda x: f"{x:.1f}")
stats["Largest file (GB)"] = stats["Largest file (GB)"].apply(lambda x: f"{x:.1f}")

stats